In [1]:
# Set project directory
%cd /content/drive/MyDrive/app_note_rag

/content/drive/MyDrive/app_note_rag


In [51]:
# Install LangChain OpenAI integration

!pip install -q langchain-openai psycopg FlagEmbedding langchain_text_splitters

In [11]:
# Set the Groq API key

import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [39]:
# Initialize the PostgreSQL database

from src.database import init_database

init_database()

✅ PostgreSQL initialized.


In [40]:
# Verify the PostgreSQL connection

from src.database import get_connection

connection = get_connection()

print("PostgreSQL connection successful.")

connection.close()

PostgreSQL connection successful.


In [72]:
# Start the FastAPI server in the background

!nohup uvicorn api.main:app --host 127.0.0.1 --port 8000 > /tmp/fastapi.log 2>&1 &

In [73]:
# Check the FastAPI server

import time

time.sleep(30)

!cat /tmp/fastapi.log

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5217.04it/s]
INFO:     Started server process [44444]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [42]:
# Test the agent API

import requests

response = requests.post(
    "http://127.0.0.1:8000/agent",
    json={
        "user_id": 4,
        "message": "What do my notes say about artificial intelligence research?",
    },
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: {"response":"Your notes describe artificial intelligence research as a multidisciplinary effort that relies on several key components:\n\n1. **Core AI Techniques**  \n   - **Machine learning** (both supervised and unsupervised) is the foundation, with a focus on learning patterns from data.  \n   - **Deep learning** and **natural language processing** are highlighted as important sub‑fields.\n\n2. **Research Practices**  \n   - **Reliable datasets** are essential for training and evaluation.  \n   - **Appropriate evaluation methods** (metrics, validation data, control of overfitting) are required to assess model performance.  \n   - **Reproducible experiments** are emphasized to validate findings and ensure that results can be independently verified.\n\n3. **Supporting Technologies**  \n   - Your notes also mention **Retrieval Augmented Generation (RAG)**, which combines information retrieval with language‑model generation. RAG systems retrieve relevant documents 

In [46]:
# Test the notes API

import requests

response = requests.get(
    "http://127.0.0.1:8000/notes",
    params={"user_id": 4},
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: [{"id":2,"title":"AI Research","content":"This note contains information about LangGraph architecture.","created_at":"2026-09-30 07:17:52.535759","updated_at":"2026-09-30 07:17:52.535759"},{"id":6,"title":"Machine Learning","content":"\nMachine learning is a field of artificial intelligence focused on learning\npatterns from data. Supervised learning uses labeled examples, while\nunsupervised learning discovers patterns without explicit labels. Model\nevaluation requires appropriate metrics, validation data, and careful control\nof overfitting.\n","created_at":"2026-09-30 20:02:13.982193","updated_at":"2026-09-30 20:02:13.982193"},{"id":7,"title":"PostgreSQL","content":"\nPostgreSQL is a powerful open-source relational database system. It supports\ntransactions, foreign keys, indexing, full-text search, and extensible data\ntypes. The pgvector extension allows PostgreSQL to store and search vector\nembeddings for similarity-based retrieval.\n","created_at":"2026-0

In [55]:
# Test note editing

import requests

response = requests.put(
    "http://127.0.0.1:8000/notes/2",
    json={
        "user_id": 4,
        "title": "AI Research Updated",
        "content": "Artificial intelligence research focuses on machine learning, deep learning, natural language processing, and reliable experimentation.",
    },
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: {"id":2,"title":"AI Research Updated","content":"Artificial intelligence research focuses on machine learning, deep learning, natural language processing, and reliable experimentation.","created_at":"2026-09-30 07:17:52.535759","updated_at":"2026-10-01 14:09:14.641912"}


In [56]:
# Test RAG after note editing

import requests

response = requests.post(
    "http://127.0.0.1:8000/agent",
    json={
        "user_id": 4,
        "message": "What does my updated AI Research note say about artificial intelligence research?",
    },
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: {"response":"Your **“AI Research Updated”** note says:\n\n> *“Artificial intelligence research focuses on machine learning, deep learning, natural language processing, and reliable experimentation.”*"}


In [60]:
# Test note deletion

import requests

response = requests.delete(
    "http://127.0.0.1:8000/notes/10",
    params={
        "user_id": 4,
    },
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: {"message":"Note deleted successfully.","note_id":10}


KeyboardInterrupt: 

In [67]:
# Install ngrok

!pip install -q pyngrok

In [68]:
# Configure ngrok authentication

from pyngrok import ngrok

ngrok.set_auth_token(input("Enter your ngrok auth token: "))

Enter your ngrok auth token: 3JOwBu0Zx9mwgraM9uR00DhVtme_4eWCjYdTWAsARSaspAHhL


In [74]:
# Create the public ngrok tunnel

from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8000)

print(f"Public API URL: {public_url}")

Public API URL: NgrokTunnel: "https://sensitive-rubber-demanding.ngrok-free.dev" -> "http://localhost:8000"


In [86]:
# Test the public API through ngrok

import requests

response = requests.get(
    "https://sensitive-rubber-demanding.ngrok-free.dev/notes?user_id=4",
    headers={"Origin": "http://localhost:3000"},
)

print("Status:", response.status_code)
print("CORS:", response.headers.get("access-control-allow-origin"))
print("Response:", response.text[:300])

Status: 200
CORS: http://localhost:3000
Response: [{"id":6,"title":"Machine Learning","content":"\nMachine learning is a field of artificial intelligence focused on learning\npatterns from data. Supervised learning uses labeled examples, while\nunsupervised learning discovers patterns without explicit labels. Model\nevaluation requires appropriate 


In [114]:
# Restart the FastAPI server

!pkill -f "uvicorn api.main:app" || true
!nohup uvicorn api.main:app --host 127.0.0.1 --port 8000 > /tmp/fastapi.log 2>&1 &

^C


In [117]:
# Check the FastAPI server

!cat /tmp/fastapi.log

In [118]:
# Test user registration

import requests

response = requests.post(
    "http://127.0.0.1:8000/register",
    json={"username": "Mohammad"},
)

print(response.status_code)
print(response.json())

200
{'user_id': 6, 'username': 'Mohammad'}


In [120]:


response = requests.post(
    "http://127.0.0.1:8000/register",
    json={"username": "Mohammad"},
)

print(response.status_code)
print(response.json())

409
{'detail': 'Username already exists.'}


In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

In [121]:
# Commit and push the latest project changes

%cd /content/drive/MyDrive/app_note_rag

!git status
!git add .
!git commit -m "feat: complete notes AI app authentication and frontend integration"
!git push origin main

/content/drive/MyDrive/app_note_rag
Refresh index: 100% (25/25), done.
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   db_backups/app_note_rag_backup.sql
	modified:   notebooks/04_agent_test.ipynb
	modified:   src/agent/graph.py
	modified:   src/agent/tools.py
	modified:   src/database.py
	modified:   src/note_service.py
	modified:   src/schema.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	api/
	notebooks/05_rag_database_setup.ipynb
	notebooks/06_ingestion_test.ipynb
	notebooks/07_retrieval_test.ipynb
	notebooks/08_reranker_test.ipynb
	notebooks/09_rag_pipeline_test.ipynb
	notebooks/10_fastapi_test.ipynb
	src/note_edit_service.py
	src/rag/

no changes added to commit (use "git add" and/or "git commit -a")
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you